# Pittsburgh Dual-Model Proximity Analysis

**Project:** Odor-Complaint Weather Risk Prediction
**Purpose:** Engineer emission-source proximity features (distance, exponential decay,
continuous wind-alignment) and compare a **weather-only** logit model (Model A) against a
**proximity-enhanced** logit model (Model B).

**Key output:** `Pittsburgh Data/model_coeffs_pittsburgh.json` — coefficients ported into the live Calvert City forecaster.

---


## Section 1 — Setup & Data Loading

Load the merged Pittsburgh hourly CSV, apply column mapping, derive zip centroids,
aggregate to a daily zip-level panel, and engineer base features.


In [ ]:
import json
import math
import warnings

import matplotlib
matplotlib.use("Agg")   # comment out if running interactively in Jupyter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings("ignore")

# Optional packages
try:
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_curve, roc_auc_score
    from sklearn.preprocessing import StandardScaler
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print("sklearn not found – CV ROC-AUC will be skipped")

try:
    import holidays
    HAS_HOLIDAYS = True
except ImportError:
    HAS_HOLIDAYS = False
    print("holidays package not found – is_holiday will be 0")


In [ ]:
DATA_PATH   = "Pittsburgh Data/open-meteo-smell-merged.csv"
PLOT_PREFIX = "Pittsburgh Data/Dual_Model_Proximity_Analysis"

COL_MAP = {
    "time":                          "datetime",
    "smell_report_count":            "complaints",
    "temperature_2m (°F)":           "temperature",
    "relative_humidity_2m (%)":      "relative_humidity",
    "wind_speed_10m (mp/h)":         "wind_speed",
    "wind_direction_10m (°)":        "wind_direction",
    "rain (inch)":                   "precipitation",
    "dew_point_2m (°F)":             "dew_point",
    "vapour_pressure_deficit (kPa)": "vapor_pressure",
    "surface_pressure (hPa)":        "atmospheric_pressure",
    "shortwave_radiation (W/m²)":    "solar_radiation",
    "sunshine_duration (s)":         "sunshine_duration",
    "boundary_layer_height (ft)":    "boundary_layer_height",
}

print(f"Loading {DATA_PATH} …")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"  Raw shape: {df_raw.shape}")

rename_actual = {k: v for k, v in COL_MAP.items() if k in df_raw.columns}
df_raw = df_raw.rename(columns=rename_actual)

numeric_cols = ["complaints", "temperature", "relative_humidity", "wind_speed",
                "wind_direction", "precipitation", "atmospheric_pressure",
                "solar_radiation", "boundary_layer_height", "smell_value_average"]
for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], errors="coerce")
df_raw["date"]     = df_raw["datetime"].dt.date
df_raw["zipcode"]  = df_raw["zipcode"].astype(str)
print(f"  Zipcodes: {df_raw['zipcode'].nunique()}")
print(f"  Date range: {df_raw['date'].min()} to {df_raw['date'].max()}")


In [ ]:
# Zip centroids
zip_centroids = (
    df_raw.groupby("zipcode")[["latitude", "longitude"]]
    .mean().reset_index()
    .rename(columns={"latitude": "lat", "longitude": "lon"})
)
print(f"Zip centroids computed for {len(zip_centroids)} zips")
zip_centroids.head()


In [ ]:
def circular_wind_mean(series):
    """Circular mean for wind direction (degrees)."""
    rads = np.radians(series.dropna())
    if len(rads) == 0:
        return np.nan
    u = np.mean(np.sin(rads))
    v = np.mean(np.cos(rads))
    return (np.degrees(np.arctan2(u, v)) + 360) % 360

# Drop true no-report padding rows (complaints==0 AND smell_value_average is NaN)
mask = (df_raw["complaints"] == 0) & (df_raw["smell_value_average"].isna())
df_filtered = df_raw[~mask].copy()
print(f"Dropped {mask.sum():,} padding rows; {len(df_filtered):,} remain")


In [ ]:
agg_spec = {
    "complaints":            "sum",
    "temperature":           ["mean", "min", "max"],
    "precipitation":         "sum",
    "wind_speed":            "mean",
    "atmospheric_pressure":  "mean",
    "solar_radiation":       "mean",
    "boundary_layer_height": "mean",
    "relative_humidity":     "mean",
    "smell_value_average":   "mean",
}
agg_spec = {k: v for k, v in agg_spec.items() if k in df_filtered.columns}

grouped   = df_filtered.groupby(["zipcode", "date"])
daily_zip = grouped.agg(agg_spec)
daily_zip.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c
                     for c in daily_zip.columns]
daily_zip = daily_zip.reset_index()

# Circular wind direction
wind_circ = grouped["wind_direction"].apply(circular_wind_mean).reset_index()
wind_circ.columns = ["zipcode", "date", "wind_direction"]
daily_zip = daily_zip.merge(wind_circ, on=["zipcode", "date"], how="left")
daily_zip.columns = [c.replace(" ", "_") for c in daily_zip.columns]

flat_rename = {
    "temperature_mean": "temperature",  "temperature_min": "temperature_min",
    "temperature_max": "temperature_max", "precipitation_sum": "precipitation",
    "wind_speed_mean": "wind_speed",    "atmospheric_pressure_mean": "atmospheric_pressure",
    "solar_radiation_mean": "solar_radiation",
    "boundary_layer_height_mean": "boundary_layer_height",
    "relative_humidity_mean": "relative_humidity",
    "smell_value_average_mean": "smell_value_average",
    "complaints_sum": "complaints",
}
daily_zip = daily_zip.rename(columns={k: v for k, v in flat_rename.items()
                                       if k in daily_zip.columns})
print(f"Daily zip panel shape: {daily_zip.shape}")
daily_zip.head()


In [ ]:
daily_zip["date"] = pd.to_datetime(daily_zip["date"])
daily_zip["diurnal_temperature_range"] = daily_zip["temperature_max"] - daily_zip["temperature_min"]
daily_zip["temperature_squared"]       = daily_zip["temperature"] ** 2

daily_zip["smell_value_average"]  = daily_zip["smell_value_average"].fillna(0)
daily_zip["weighted_odor_burden"] = daily_zip["complaints"] * daily_zip["smell_value_average"]
wob_mean = daily_zip["weighted_odor_burden"].mean()
daily_zip["is_odor_event"] = (daily_zip["weighted_odor_burden"] > wob_mean).astype(int)
print(f"ORI event rate: {daily_zip['is_odor_event'].mean():.3f}  (WOB threshold={wob_mean:.2f})")

# Day-of-week dummies (Monday = reference)
dow_map = {1: "tue", 2: "wed", 3: "thu", 4: "fri", 5: "sat", 6: "sun"}
for num, name in dow_map.items():
    daily_zip[f"dow_{name}"] = (daily_zip["date"].dt.dayofweek == num).astype(int)

if HAS_HOLIDAYS:
    us_hols = holidays.US()
    daily_zip["is_holiday"] = daily_zip["date"].apply(lambda d: int(d in us_hols))
else:
    daily_zip["is_holiday"] = 0

print(f"Final daily_zip shape: {daily_zip.shape}")


## Section 2 — Proximity Feature Engineering

For each emission source, compute:
- **Haversine distance** (miles) from source to zip centroid
- **Exponential decay** `exp(−0.02 × dist)` — exposure proxy
- **Continuous wind alignment** (0–1) — 1 when wind blows directly from source to receptor

Then aggregate across sources to get `multi_source_exposure` and `wind_align_weighted`.


In [ ]:
EMISSION_SOURCES = {
    "Clairton_Coke_Works": (40.2974, -79.8809),
    "Edgar_Thomson_Works":  (40.3922, -79.8550),
    "Irvin_Works":          (40.3644, -79.8944),
}

def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = (math.sin(dphi/2)**2
         + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2)
    return R * 2 * math.asin(math.sqrt(a))

def bearing(src_lat, src_lon, dst_lat, dst_lon):
    dy = dst_lat - src_lat
    dx = (dst_lon - src_lon) * math.cos(math.radians(src_lat))
    return (math.degrees(math.atan2(dx, dy)) + 360) % 360

def continuous_wind_alignment(wind_from_deg, bearing_deg):
    """
    Returns a value in [0, 1].
    1 = wind blowing directly from source toward receptor.
    """
    wind_toward = (wind_from_deg + 180) % 360
    angle_diff  = math.radians(wind_toward - bearing_deg)
    return (1 + math.cos(angle_diff)) / 2


In [ ]:
# Merge zip centroids
daily_zip = daily_zip.merge(zip_centroids[["zipcode", "lat", "lon"]],
                             on="zipcode", how="left")

for src_name, (src_lat, src_lon) in EMISSION_SOURCES.items():
    print(f"  Computing features for {src_name} …")
    daily_zip[f"dist_{src_name}"] = daily_zip.apply(
        lambda r: haversine_miles(src_lat, src_lon, r["lat"], r["lon"]), axis=1)
    daily_zip[f"exp02_{src_name}"] = np.exp(-0.02 * daily_zip[f"dist_{src_name}"])
    daily_zip[f"bearing_{src_name}"] = daily_zip.apply(
        lambda r: bearing(src_lat, src_lon, r["lat"], r["lon"]), axis=1)
    daily_zip[f"wind_align_{src_name}"] = daily_zip.apply(
        lambda r: continuous_wind_alignment(
            r["wind_direction"], r[f"bearing_{src_name}"]), axis=1)

exp02_cols      = [f"exp02_{s}"      for s in EMISSION_SOURCES]
wind_align_cols = [f"wind_align_{s}" for s in EMISSION_SOURCES]
dist_cols       = [f"dist_{s}"       for s in EMISSION_SOURCES]

daily_zip["multi_source_exposure"] = daily_zip[exp02_cols].sum(axis=1)
total_exp = daily_zip[exp02_cols].sum(axis=1).replace(0, np.nan)
weighted_num = sum(daily_zip[f"exp02_{s}"] * daily_zip[f"wind_align_{s}"]
                   for s in EMISSION_SOURCES)
daily_zip["wind_align_weighted"] = weighted_num / total_exp
daily_zip["dist_nearest"]        = daily_zip[dist_cols].min(axis=1)

print(f"multi_source_exposure  : {daily_zip['multi_source_exposure'].min():.3f} – {daily_zip['multi_source_exposure'].max():.3f}")
print(f"wind_align_weighted    : {daily_zip['wind_align_weighted'].min():.3f} – {daily_zip['wind_align_weighted'].max():.3f}")
print(f"dist_nearest           : {daily_zip['dist_nearest'].min():.1f} – {daily_zip['dist_nearest'].max():.1f} miles")


## Section 3 — Binned Distribution Plots

Bin each proximity feature into 8 quantile groups and plot mean daily complaints per bin
with standard-error bars.


In [ ]:
features_to_plot = {
    "dist_nearest":          "Distance to Nearest Source (miles)",
    "multi_source_exposure": "Multi-Source Exposure (Σ exp(−0.02·dist))",
    "wind_align_weighted":   "Exposure-Weighted Wind Alignment (0–1)",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Pittsburgh: Mean Daily Complaints by Proximity Feature Bin",
             fontsize=13, fontweight="bold")

for ax, (feat, label) in zip(axes, features_to_plot.items()):
    try:
        bins = pd.qcut(daily_zip[feat], q=8, duplicates="drop")
    except Exception:
        bins = pd.cut(daily_zip[feat], bins=8)
    grp = daily_zip.groupby(bins)["complaints"].agg(["mean", "sem"]).dropna()
    x   = range(len(grp))
    ax.bar(x, grp["mean"], yerr=grp["sem"], capsize=4,
           color="steelblue", alpha=0.8, edgecolor="white")
    ax.set_xticks(list(x))
    ax.set_xticklabels([str(b) for b in grp.index], rotation=45, ha="right", fontsize=7)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Mean Daily Complaints", fontsize=9)
    ax.set_title(feat.replace("_", " ").title(), fontsize=10)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section3_bins.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved.")


## Section 4 — Feature Importance Screening

Compare four model specifications (weather-only, +proximity, +alignment, +both)
using both Logit and Poisson GLMs.  Displays Pseudo-R², AIC, and p-values for the
two new proximity features.


In [ ]:
weather_vars = [
    "temperature", "temperature_squared", "solar_radiation",
    "relative_humidity", "wind_speed", "precipitation",
    "diurnal_temperature_range", "boundary_layer_height", "atmospheric_pressure",
    "dow_tue", "dow_wed", "dow_thu", "dow_fri", "dow_sat", "dow_sun", "is_holiday",
]
weather_vars = [v for v in weather_vars if v in daily_zip.columns]

model_specs = {
    "Weather Only":     weather_vars,
    "+Proximity":       weather_vars + ["multi_source_exposure"],
    "+Wind Alignment":  weather_vars + ["wind_align_weighted"],
    "+Both":            weather_vars + ["multi_source_exposure", "wind_align_weighted"],
}

target_logit   = "is_odor_event"
target_poisson = "complaints"
results_table  = []

for spec_name, feat_list in model_specs.items():
    sub = daily_zip[feat_list + [target_logit, target_poisson]].dropna()
    X   = sm.add_constant(sub[feat_list])
    y_b = sub[target_logit]
    y_p = sub[target_poisson]

    try:
        lf   = sm.Logit(y_b, X).fit(disp=0, method="bfgs", maxiter=300)
        lr2  = lf.prsquared; aic = lf.aic
        p_e  = lf.pvalues.get("multi_source_exposure", np.nan)
        p_a  = lf.pvalues.get("wind_align_weighted",  np.nan)
    except:
        lr2 = aic = p_e = p_a = np.nan

    try:
        pf   = sm.GLM(y_p, X, family=sm.families.Poisson()).fit(disp=0)
        p_r2 = 1 - pf.deviance / pf.null_deviance; p_aic = pf.aic
    except:
        p_r2 = p_aic = np.nan

    results_table.append({
        "Model":             spec_name,
        "Logit_PseudoR2":   round(lr2,  4),
        "Logit_AIC":        round(aic,  1),
        "Poisson_PseudoR2": round(p_r2, 4),
        "Poisson_AIC":      round(p_aic,1),
        "p(exposure)":      f"{p_e:.4f}" if not np.isnan(p_e) else "—",
        "p(alignment)":     f"{p_a:.4f}" if not np.isnan(p_a) else "—",
    })

pd.DataFrame(results_table)


## Section 5 — Dual-Model Full Comparison

### A. Fit both logit models on the complete-case sample


In [ ]:
prox_vars = weather_vars + ["multi_source_exposure", "wind_align_weighted"]
sub_full  = daily_zip[list(set(prox_vars + [target_logit, target_poisson]))].dropna()
print(f"Complete-case rows: {len(sub_full):,}")

X_wa  = sm.add_constant(sub_full[weather_vars])
X_pe  = sm.add_constant(sub_full[prox_vars])
y_bin = sub_full[target_logit]

logit_weather_only       = sm.Logit(y_bin, X_wa).fit(disp=0, method="bfgs", maxiter=300)
logit_proximity_enhanced = sm.Logit(y_bin, X_pe).fit(disp=0, method="bfgs", maxiter=300)
poisson_wa               = sm.GLM(sub_full[target_poisson], X_wa,
                                   family=sm.families.Poisson()).fit(disp=0)
poisson_pe               = sm.GLM(sub_full[target_poisson], X_pe,
                                   family=sm.families.Poisson()).fit(disp=0)

delta_r2  = logit_proximity_enhanced.prsquared - logit_weather_only.prsquared
delta_aic = logit_weather_only.aic - logit_proximity_enhanced.aic
print(f"ΔPseudo-R²  (B − A) : {delta_r2:+.4f}")
print(f"ΔAIC        (A − B) : {delta_aic:+.1f}  (positive = Model B better)")


### B. Coefficient bar chart — top 14 features by magnitude

In [ ]:
def top_n_coefs(fit_result, n=12):
    coefs = fit_result.params.drop("const", errors="ignore")
    return coefs.reindex(coefs.abs().nlargest(n).index)

coefs_b   = top_n_coefs(logit_proximity_enhanced, n=12)
coefs_a   = top_n_coefs(logit_weather_only,        n=12)
all_feat  = list(dict.fromkeys(list(coefs_b.index) + list(coefs_a.index)))[:14]
coefs_a_v = [logit_weather_only.params.get(f, 0)       for f in all_feat]
coefs_b_v = [logit_proximity_enhanced.params.get(f, 0) for f in all_feat]

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = np.arange(len(all_feat)); bar_h = 0.35
ax.barh(y_pos + bar_h/2, coefs_b_v, bar_h, label="Model B (Proximity-Enhanced)",
        color="steelblue", alpha=0.85)
ax.barh(y_pos - bar_h/2, coefs_a_v, bar_h, label="Model A (Weather Only)",
        color="coral",    alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([f.replace("_", " ") for f in all_feat], fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Logit Coefficient", fontsize=11)
ax.set_title("Logit Coefficients: Weather-Only vs Proximity-Enhanced (top 14)", fontsize=11)
ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section5B_coefs.png", dpi=120, bbox_inches="tight")
plt.show()


### C. Predicted ORI probability distributions

In [ ]:
pred_a = logit_weather_only.predict(X_wa)
pred_b = logit_proximity_enhanced.predict(X_pe)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(pred_a, bins=60, alpha=0.55, label="Model A – Weather Only",
        color="coral",     density=True, edgecolor="white")
ax.hist(pred_b, bins=60, alpha=0.55, label="Model B – Proximity-Enhanced",
        color="steelblue", density=True, edgecolor="white")
ax.set_xlabel("Predicted ORI Probability", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Pittsburgh: Predicted ORI Probability Distributions", fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section5C_distributions.png", dpi=120, bbox_inches="tight")
plt.show()


### D. ROC curves + optional 5-fold CV AUC

In [ ]:
fpr_a, tpr_a, _ = roc_curve(y_bin, pred_a)
fpr_b, tpr_b, _ = roc_curve(y_bin, pred_b)
auc_a = roc_auc_score(y_bin, pred_a)
auc_b = roc_auc_score(y_bin, pred_b)

if HAS_SKLEARN:
    scaler = StandardScaler()
    try:
        cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        lr_clf = LogisticRegression(max_iter=500, solver="lbfgs")
        cv_auc_a = cross_val_score(lr_clf,
                                   scaler.fit_transform(sub_full[weather_vars].fillna(0)),
                                   y_bin.values, cv=cv, scoring="roc_auc").mean()
        cv_auc_b = cross_val_score(lr_clf,
                                   scaler.fit_transform(sub_full[prox_vars].fillna(0)),
                                   y_bin.values, cv=cv, scoring="roc_auc").mean()
        print(f"5-fold CV ROC-AUC  A: {cv_auc_a:.4f}   B: {cv_auc_b:.4f}")
    except Exception as e:
        print(f"CV AUC failed: {e}")
        cv_auc_a = cv_auc_b = None
else:
    cv_auc_a = cv_auc_b = None

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_a, tpr_a, color="coral",     lw=2, label=f"Model A (AUC={auc_a:.3f})")
ax.plot(fpr_b, tpr_b, color="steelblue", lw=2, label=f"Model B (AUC={auc_b:.3f})")
ax.plot([0,1],[0,1], "k--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate",  fontsize=11)
ax.set_title("Pittsburgh ORI Logit — ROC Curves", fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section5D_roc.png", dpi=120, bbox_inches="tight")
plt.show()


### E. Full Model B statsmodels summary

In [ ]:
print(logit_proximity_enhanced.summary())
print(f"\nKey proximity coefficients:")
print(f"  multi_source_exposure: {logit_proximity_enhanced.params['multi_source_exposure']:.6f}  "
      f"p={logit_proximity_enhanced.pvalues['multi_source_exposure']:.4f}")
print(f"  wind_align_weighted  : {logit_proximity_enhanced.params['wind_align_weighted']:.6f}  "
      f"p={logit_proximity_enhanced.pvalues['wind_align_weighted']:.4f}")


## Section 6 — Export Coefficients to JSON

> **Critical:** These coefficients are ported into the live Calvert City forecaster.
> `dow_*` and `is_holiday` flags are set to 0 for Calvert City predictions (de-biasing).


In [ ]:
coeffs_wa = logit_weather_only.params.to_dict()
coeffs_pe = logit_proximity_enhanced.params.to_dict()

output = {
    "city": "Pittsburgh",
    "note": (
        "Coefficients from logit model trained on Pittsburgh zip-day panel "
        "(de-biased: dow/holiday vars set to 0 for Calvert City predictions). "
        "Proximity-enhanced includes multi_source_exposure and wind_align_weighted."
    ),
    "weather_only": {k: float(v) for k, v in coeffs_wa.items()},
    "proximity_enhanced": {k: float(v) for k, v in coeffs_pe.items()},
    "model_metrics": {
        "weather_only_aic":             float(logit_weather_only.aic),
        "proximity_enhanced_aic":       float(logit_proximity_enhanced.aic),
        "delta_aic":                    float(logit_weather_only.aic - logit_proximity_enhanced.aic),
        "weather_only_pseudo_r2":       float(logit_weather_only.prsquared),
        "proximity_enhanced_pseudo_r2": float(logit_proximity_enhanced.prsquared),
        "weather_only_auc":             float(auc_a),
        "proximity_enhanced_auc":       float(auc_b),
    },
}
if cv_auc_a is not None:
    output["model_metrics"]["weather_only_cv_auc"]       = float(cv_auc_a)
    output["model_metrics"]["proximity_enhanced_cv_auc"] = float(cv_auc_b)

json_path = "Pittsburgh Data/model_coeffs_pittsburgh.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {json_path}")
print(f"Key proximity coefficients:")
print(f"  multi_source_exposure: {coeffs_pe.get('multi_source_exposure', 'N/A')}")
print(f"  wind_align_weighted  : {coeffs_pe.get('wind_align_weighted',   'N/A')}")
print(f"\nΔPseudo-R² (B−A): {output['model_metrics']['proximity_enhanced_pseudo_r2'] - output['model_metrics']['weather_only_pseudo_r2']:+.4f}")
print(f"ΔAIC        (A−B): {output['model_metrics']['delta_aic']:+.1f}")
